# Extract 
- From Azure Blob storage

In [ ]:
import os
from io import BytesIO

import pandas as pd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")


def load(container_name, file_path):
    """Read one CSV file from Azure Blob Storage and return it as a DataFrame.
    
    Args:
        container_name (str): The name of the blob container
        file_path (str): The path to the file within the container
        
    Returns:
        pd.DataFrame: The loaded CSV file as a DataFrame
        
    Raises:
        ValueError: If AZURE_STORAGE_CONNECTION_STRING is not set
    """
    if not connection_string:
        raise ValueError(
            "AZURE_STORAGE_CONNECTION_STRING is not set in the environment."
        )

    print(f"Loading: {file_path}")

    blob_service_client = BlobServiceClient.from_connection_string(
        connection_string
    )
    container_client = blob_service_client.get_container_client(container_name)

    blob_client = container_client.get_blob_client(file_path)
    blob_data = blob_client.download_blob().readall()

    df = pd.read_csv(BytesIO(blob_data))
    print(f"File successful: {file_path}")
    return df


Extract files(container: baraa)
- source_crm
    - cust_info.csv
    - prd_info.csv
    - sales_details.csv
- source_erp
    - CUST_AZ12.csv
    - LOC_A101.csv
    - PX_CAT_G1V2.csv

In [ ]:
def extract_all():
    """Extract all data files from Azure Blob Storage.
    
    Loads 6 CSV files from Azure Blob Storage:
    - CRM: customer info, product info, sales details
    - ERP: customer data, location data, product category data
    
    Returns:
        dict: Dictionary containing all loaded DataFrames, or None if extraction fails
    """
    try:
        global crm_cust_info, crm_prd_info, crm_sales_details
        global erp_cust_az12, erp_loc_a101, erp_px_cat_g1v2

        crm_cust_info = load("baraa", "source_crm/cust_info.csv")
        crm_prd_info = load("baraa", "source_crm/prd_info.csv")
        crm_sales_details = load("baraa", "source_crm/sales_details.csv")
        erp_cust_az12 = load("baraa", "source_erp/CUST_AZ12.csv")
        erp_loc_a101 = load("baraa", "source_erp/LOC_A101.csv")
        erp_px_cat_g1v2 = load("baraa", "source_erp/PX_CAT_G1V2.csv")

        print("✓ All CSV files loaded successfully")
        return {
            "crm_cust_info": crm_cust_info,
            "crm_prd_info": crm_prd_info,
            "crm_sales_details": crm_sales_details,
            "erp_cust_az12": erp_cust_az12,
            "erp_loc_a101": erp_loc_a101,
            "erp_px_cat_g1v2": erp_px_cat_g1v2,
        }
    except Exception as e:
        print(f"✗ File extraction failed: {e}")
        return None


all_data = extract_all()


Loading: source_crm/cust_info.csv
File successful: source_crm/cust_info.csv
Loading: source_crm/prd_info.csv
File successful: source_crm/prd_info.csv
Loading: source_crm/sales_details.csv
File successful: source_crm/sales_details.csv
Loading: source_erp/CUST_AZ12.csv
File successful: source_erp/CUST_AZ12.csv
Loading: source_erp/LOC_A101.csv
File successful: source_erp/LOC_A101.csv
Loading: source_erp/PX_CAT_G1V2.csv
File successful: source_erp/PX_CAT_G1V2.csv
All csv files are loaded


# Transform
- Clean and standardize the data from all sources
- Handle missing values, data type conversions, and data validation


In [ ]:
def clean_customer_data(crm_cust_info):
    """Clean and transform customer data.
    
    Args:
        crm_cust_info (pd.DataFrame): Raw customer data from CRM
        
    Returns:
        pd.DataFrame: Cleaned customer data with standardized values
    """

    # 1. Copy source DataFrame
    df = crm_cust_info.copy()

    # 2. Remove duplicates and rows missing required IDs
    df = df.drop_duplicates()
    df = df.dropna(subset=["cst_id", "cst_key"])

    # 3. Convert data types
    df["cst_id"] = df["cst_id"].astype(int)

    df["cst_create_date"] = pd.to_datetime(
        df["cst_create_date"],
        errors="coerce"
    )

    # 4. Keep the latest record for each customer
    df = (
        df.sort_values("cst_create_date")
          .drop_duplicates(
              subset="cst_id",
              keep="last"
          )
    )

    # 5. Clean customer names
    name_columns = [
        "cst_firstname2",
        "cst_lastname2"
    ]

    df[name_columns] = df[name_columns].apply(
        lambda col: col.str.strip()
    )

    # 6. Rename columns
    df = df.rename(columns={
        "cst_firstname2": "cst_firstname",
        "cst_lastname2": "cst_lastname"
    })

    # 7. Convert coded values to readable values
    df["cst_marital_status"] = df["cst_marital_status"].replace({
        "M": "Married",
        "S": "Single"
    })

    df["cst_gndr"] = df["cst_gndr"].replace({
        "M": "Male",
        "F": "Female"
    })

    # 8. Replace remaining missing values
    df = df.fillna("N/A")

    return df


In [4]:
df_customer = clean_customer_data(crm_cust_info)

In [ ]:
def clean_product_data(crm_prd_info):
    """Clean and transform product data.
    
    Args:
        crm_prd_info (pd.DataFrame): Raw product data from CRM
        
    Returns:
        pd.DataFrame: Cleaned product data with category extraction and date validation
    """
    # 1. Copy source DataFrame and remove duplicates
    df = crm_prd_info.copy()
    df = df.drop_duplicates()
    df = df.dropna(subset=["prd_id", "prd_key"])

    # 2. Extract category ID from product key
    df["cat_id"] = df["prd_key"].str[:5].str.replace("-", "_")
    df["prd_key"] = df["prd_key"].str[6:]

    # 3. Move category ID column to position after product ID
    cat_id = df.pop("cat_id")
    df.insert(df.columns.get_loc("prd_id") + 1, "cat_id", cat_id)

    # 4. Fill missing costs with 0
    df["prd_cost"] = df["prd_cost"].fillna(0)

    # 5. Standardize product line codes to readable names
    prd_line_mapping = {
        "M": "Mountain",
        "R": "Road",
        "S": "Other Sale",
        "T": "Touring"
    }
    df["prd_line"] = (
        df["prd_line"]
        .str.upper()
        .str.strip()
        .map(prd_line_mapping)
        .fillna("N/A")
    )

    # 6. Convert date columns to datetime type
    df["prd_start_dt"] = pd.to_datetime(df["prd_start_dt"])
    df["prd_end_dt"] = pd.to_datetime(df["prd_end_dt"])

    # 7. Sort by product key and start date
    df = df.sort_values(["prd_key", "prd_start_dt"])

    # 8. Validate and fix date ranges
    # Find rows where start date is after end date
    is_broken = df["prd_start_dt"] > df["prd_end_dt"]

    # Calculate next product start date minus 1 day (use as end date for broken rows)
    next_start_minus_1 = (
        df.groupby("prd_key")["prd_start_dt"]
        .shift(-1)
        - pd.Timedelta(days=1)
    )

    # Fix only the broken rows
    df.loc[is_broken, "prd_end_dt"] = next_start_minus_1[is_broken]

    return df


## Additional Data Transformations

The following data sources also need to be cleaned and prepared:
- **Sales Details**: Transaction data from CRM
- **ERP Sources**: Customer, location, and product category data

In [ ]:
df_product = clean_product_data(crm_prd_info)


In [ ]:
# Validate: Check for any remaining date range violations
validation_result = df_product[df_product["prd_start_dt"] > df_product["prd_end_dt"]]
print(f"Rows with start_date > end_date: {len(validation_result)}")
validation_result


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
2,212,AC_HE,HL-U509-R,Sport-100 Helmet- Red,12.0,Other Sale,2011-07-01,2007-12-28
3,213,AC_HE,HL-U509-R,Sport-100 Helmet- Red,14.0,Other Sale,2012-07-01,2008-12-27
5,215,AC_HE,HL-U509,Sport-100 Helmet- Black,12.0,Other Sale,2011-07-01,2007-12-28
6,216,AC_HE,HL-U509,Sport-100 Helmet- Black,14.0,Other Sale,2012-07-01,2008-12-27
8,218,CL_SO,SO-B909-M,Mountain Bike Socks- M,3.0,Mountain,2011-07-01,2007-12-28
...,...,...,...,...,...,...,...,...
254,464,CL_GL,GL-H102-M,Half-Finger Gloves- M,10.0,Other Sale,2012-07-01,2008-12-27
256,466,CL_GL,GL-H102-L,Half-Finger Gloves- L,10.0,Other Sale,2012-07-01,2008-12-27
258,468,CL_GL,GL-F110-S,Full-Finger Gloves- S,16.0,Mountain,2012-07-01,2008-12-27
259,469,CL_GL,GL-F110-M,Full-Finger Gloves- M,16.0,Mountain,2012-07-01,2008-12-27


In [ ]:
# Display product data statistics
print("\n" + "=" * 50)
print("PRODUCT DATA - Information Summary")
print("=" * 50)
print(df_product.info())
print("\n" + "-" * 50)
print("NULL Values Count")
print("-" * 50)
print(df_product.isnull().sum())
print("\n" + "-" * 50)
print("Unique Values Count")
print("-" * 50)
print(df_product.nunique())



 Information
<class 'pandas.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   prd_id        397 non-null    int64  
 1   prd_key       397 non-null    str    
 2   prd_nm        397 non-null    str    
 3   prd_cost      395 non-null    float64
 4   prd_line      380 non-null    str    
 5   prd_start_dt  397 non-null    str    
 6   prd_end_dt    200 non-null    str    
 7   cat_id        397 non-null    str    
dtypes: float64(1), int64(1), str(6)
memory usage: 24.9 KB
None

 Number of NULL data
prd_id            0
prd_key           0
prd_nm            0
prd_cost          2
prd_line         17
prd_start_dt      0
prd_end_dt      197
cat_id            0
dtype: int64

 Number of unique data
prd_id          397
prd_key         295
prd_nm          295
prd_cost        108
prd_line          4
prd_start_dt      4
prd_end_dt        2
cat_id           37
dtype: int64


In [ ]:
# Display first 10 rows of product data
print("First 10 rows of product data:")
df_product.head(10)


,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,cat_id
0,210,FR-R92B-58,HL Road Frame - Black- 58,NaN,R,2003-07-01,NaN,CO_RF
1,211,FR-R92R-58,HL Road Frame - Red- 58,NaN,R,2003-07-01,NaN,CO_RF
2,212,HL-U509-R,Sport-100 Helmet- Red,12.0,S,2011-07-01,2007-12-28,AC_HE
3,213,HL-U509-R,Sport-100 Helmet- Red,14.0,S,2012-07-01,2008-12-27,AC_HE
4,214,HL-U509-R,Sport-100 Helmet- Red,13.0,S,2013-07-01,NaN,AC_HE
5,215,HL-U509,Sport-100 Helmet- Black,12.0,S,2011-07-01,2007-12-28,AC_HE
6,216,HL-U509,Sport-100 Helmet- Black,14.0,S,2012-07-01,2008-12-27,AC_HE
7,217,HL-U509,Sport-100 Helmet- Black,13.0,S,2013-07-01,NaN,AC_HE
8,218,SO-B909-M,Mountain Bike Socks- M,3.0,M,2011-07-01,2007-12-28,CL_SO
9,219,SO-B909-L,Mountain Bike Socks- L,3.0,M,2011-07-01,2007-12-28,CL_SO


In [ ]:
# Display last 10 rows of product data
print("Last 10 rows of product data:")
df_product.tail(10)


,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,cat_id
387,597,BK-M18B-42,Mountain-500 Black- 42,295.0,M,2013-07-01,NaN,BI_MB
388,598,BK-M18B-44,Mountain-500 Black- 44,295.0,M,2013-07-01,NaN,BI_MB
389,599,BK-M18B-48,Mountain-500 Black- 48,295.0,M,2013-07-01,NaN,BI_MB
390,600,BK-M18B-52,Mountain-500 Black- 52,295.0,M,2013-07-01,NaN,BI_MB
391,601,BB-7421,LL Bottom Bracket,24.0,NaN,2013-07-01,NaN,CO_BB
392,602,BB-8107,ML Bottom Bracket,45.0,NaN,2013-07-01,NaN,CO_BB
393,603,BB-9108,HL Bottom Bracket,54.0,NaN,2013-07-01,NaN,CO_BB
394,604,BK-R19B-44,Road-750 Black- 44,344.0,R,2013-07-01,NaN,BI_RB
395,605,BK-R19B-48,Road-750 Black- 48,344.0,R,2013-07-01,NaN,BI_RB
396,606,BK-R19B-52,Road-750 Black- 52,344.0,R,2013-07-01,NaN,BI_RB


# Load
- Load cleaned data into MySQL data warehouse
- Ready for further analysis and reporting

## Summary

**Cleaned Data Available for Loading:**
- `df_customer`: Customer master data (deduplicated, standardized)
- `df_product`: Product master data with category extraction (date-validated)

In [ ]:
# Final Data Quality Summary
print("\n" + "=" * 60)
print("ETL PROCESS COMPLETE - DATA QUALITY SUMMARY")
print("=" * 60)

print(f"\n✓ Customer Data:")
print(f"  - Rows: {len(df_customer)}")
print(f"  - Columns: {len(df_customer.columns)}")
print(f"  - Missing values: {df_customer.isnull().sum().sum()}")

print(f"\n✓ Product Data:")
print(f"  - Rows: {len(df_product)}")
print(f"  - Columns: {len(df_product.columns)}")
print(f"  - Missing values: {df_product.isnull().sum().sum()}")
print(f"  - Date range violations: {len(df_product[df_product['prd_start_dt'] > df_product['prd_end_dt']])}")

print("\n" + "=" * 60)
print("Ready for loading into MySQL data warehouse")
print("=" * 60)
